# `iterative_feature_set_growth`

## Purpose

Grow candidate feature sets by iteratively adding features that significantly improve performance.

## Previous notebook

`collinearity_analysis`

## Next notebook

`candidate_feature_set_trimming`

# Imports

In [1]:
import numpy as np
import pandas as pd

import os
from datetime import datetime
import pickle

import warnings

import matplotlib.pyplot as plt
import seaborn as sns

from statistics import median

from scipy.stats import ttest_rel, t

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix, precision_score, recall_score, average_precision_score, RocCurveDisplay, accuracy_score

import prepare_data

import cutpoint_analysis
os.chdir(original_dir)

warnings.filterwarnings("ignore")

import mlflow

# Load data

In [2]:
train_cohort_ll = prepare_data.load_and_process_cohort('train', 'latest')

train_cohort_med_imp = prepare_data.load_and_process_cohort('train', 'median')

In [3]:
def compare_features_across_dfs(df1, df2, suffix_1='_1', suffix_2='_2', join_cols=['pdc_eid', 'pdc_hid', 'pdc_pid']):
    rename_dict_1 = dict()
    rename_dict_2 = dict()
    
    for col in df1.columns:
        if col not in join_cols:
            rename_dict_1[col] = col + suffix_1
            rename_dict_2[col] = col + suffix_2
            
    renamed_df1 = df1.rename(columns=rename_dict_1).copy()
    renamed_df2 = df2.rename(columns=rename_dict_2).copy()
    
    merged_df = renamed_df1.merge(
        renamed_df2,
        on=join_cols,
        how='inner')
    
    for col in renamed_df1.columns:
        if col not in join_cols:
            try:
                merged_df[col.replace(suffix_1, '_diff')] = merged_df[col] - merged_df[col.replace(suffix_1, suffix_2)]
            except:
                pass
                
    return merged_df[[col for col in merged_df.columns if 'mean_diff' in col]].describe()

In [4]:
compare_features_across_dfs(train_cohort_ll, train_cohort_med_imp)

,dbp_mean_diff,fio2_mean_diff,map_mean_diff,mbp_mean_diff,pox_mean_diff,pulse_mean_diff,resp_rate_mean_diff,sbp_mean_diff,temp_mean_diff,weight_mean_diff,...,PLTS_mean_diff,PO2_mean_diff,POTASSIUM_mean_diff,PROCALCITONIN_mean_diff,PROTEIN_mean_diff,PT_mean_diff,PTT_mean_diff,RDW_mean_diff,SODIUM_mean_diff,WBC_mean_diff
count,76760.0,76760.000000,76760.0,76760.0,76760.0,76760.000000,76760.0,76760.0,76760.0,76760.000000,...,76760.000000,76760.000000,76760.000000,7.676000e+04,7.676000e+04,7.676000e+04,7.676000e+04,7.676000e+04,76760.000000,7.676000e+04
mean,0.0,-0.000034,0.0,0.0,0.0,-0.000004,0.0,0.0,0.0,0.000006,...,0.020462,0.013914,0.011423,8.800319e-04,2.032088e-02,-2.562863e-03,1.737013e-03,1.406376e-02,-0.001416,1.068648e-03
std,0.0,0.000035,0.0,0.0,0.0,0.000007,0.0,0.0,0.0,0.000005,...,0.079508,0.082054,0.058887,2.203749e-02,7.966145e-02,2.726854e-02,3.634731e-02,7.985739e-02,0.032794,4.125964e-02
min,0.0,-0.000070,0.0,0.0,0.0,-0.000017,0.0,0.0,0.0,0.000000,...,-0.238046,-0.124514,-0.333333,-9.069438e-03,-5.211268e-01,-1.613300e-01,-9.633313e-02,-1.057269e-01,-0.371773,-1.350675e-03
25%,0.0,-0.000070,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,...,0.000000,0.000000,0.000000,-1.734723e-18,2.775558e-16,0.000000e+00,-4.163336e-17,-2.498002e-16,0.000000,0.000000e+00
50%,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000010,...,0.000000,0.000000,0.000000,-1.734723e-18,2.775558e-16,2.775558e-16,-4.163336e-17,-2.498002e-16,0.000000,0.000000e+00
75%,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000010,...,0.000000,0.000000,0.000000,-1.734723e-18,2.775558e-16,2.775558e-16,-4.163336e-17,0.000000e+00,0.000000,4.336809e-19
max,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000010,...,1.201663,1.192607,0.866667,1.589891e+00,8.028169e-01,1.740148e+00,1.492853e+00,1.506608e+00,0.413081,4.143782e+00


# Load results of previous notebooks

Single-feature model performance:

In [5]:
logreg_single_feature_metrics = pd.read_csv('single_feature_logreg_performance_results_latest_lab_imp.csv')
svc_single_feature_metrics = pd.read_csv('single_feature_svc_performance_results_latest_lab_imp.csv')

logreg_single_feature_metrics['auroc_logreg'] = logreg_single_feature_metrics['auroc'].copy()
logreg_single_feature_metrics['auprc_logreg'] = logreg_single_feature_metrics['auprc'].copy()
logreg_single_feature_metrics.drop(['auroc', 'auprc'], axis=1, inplace=True)

svc_single_feature_metrics['auroc_svc'] = svc_single_feature_metrics['auroc'].copy()
svc_single_feature_metrics['auprc_svc'] = svc_single_feature_metrics['auprc'].copy()
svc_single_feature_metrics.drop(['auroc', 'auprc', 'model_type'], axis=1, inplace=True)

combined_single_feature_metrics = svc_single_feature_metrics.merge(
    logreg_single_feature_metrics,
    on='input_feature', 
    how='outer'
)

del(logreg_single_feature_metrics)
del(svc_single_feature_metrics)

combined_single_feature_metrics.head()

,input_feature,auroc_svc,auprc_svc,auroc_logreg,auprc_logreg
0,ageatadmission,0.383238,0.023236,0.600715,0.109145
1,ALBUMIN_max,0.499085,0.032375,0.499244,0.032555
2,ALBUMIN_mean,0.501799,0.028491,0.498402,0.033431
3,ALBUMIN_median,0.498230,0.033083,0.498411,0.033281
4,ALBUMIN_min,0.501599,0.028378,0.498568,0.034995


Features to exclude:

In [6]:
with open('pickle/removed_collinear_features.pickle', 'rb') as infile:
    removed_features = pickle.load(infile)
    
print('# of features to exclude: %d' % len(removed_features))

# of features to exclude: 80


# Define functions

## Train logistic regression model and return cross val scores

In [7]:
def train_logreg_cv_from_feature_set(data_df, 
                                     feature_set, 
                                     scoring_metric,
                                     n_splits=10,
                                     n_jobs=10,
                                     max_iter=10000,
                                     random_state=343,
                                     scorer=None):
    X = data_df[feature_set].to_numpy()
    y = data_df['aki_72hrs_any'].to_numpy()

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    model = LogisticRegression(class_weight='balanced',
                               max_iter=max_iter)

    cv_scores = cross_val_score(model, 
                                X, y, 
                                cv=cv, 
                                scoring=scoring_metric,
                                n_jobs=n_jobs)

    return cv_scores

## Train SVC model and return cross val scores

In [8]:
def train_svc_cv_from_feature_set(data_df, 
                                 feature_set, 
                                 scoring_metric,
                                  svc_kernel='rbf',
                                 n_splits=10,
                                 n_jobs=10,
                                 max_iter=5000,
                                 random_state=343,
                                 scorer=None):
    X = data_df[feature_set].to_numpy()
    y = data_df['aki_72hrs_any'].to_numpy()

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    model = SVC(class_weight='balanced',
                kernel=svc_kernel,
                max_iter=max_iter,
                probability=True)

    cv_scores = cross_val_score(model, 
                                X, y, 
                                cv=cv, 
                                scoring=scoring_metric,
                                n_jobs=n_jobs)

    return cv_scores

## Perform hypothesis test to compare cross-val scores

Define a function to perform a Wilcoxon signed-rank hypothesis test on two sets of cross-validated AUC scores to check if score set 2 is significantly better than score set 1. With $\Delta AUC := AUC_2 - AUC_1$, the hypothesis being tested is:

$H_0$: $\Delta AUC \leq 0$ (equivalent to "score set 2 is not significantly better than score set 1").

$H_1$: $\Delta AUC > 0$ (equivalent to "score set 2 is significantly better than score set 1").

In [9]:
def cv_auc_hypothesis_test(score_set_1, score_set_2, alternative_hypothesis='greater'):
    if len(score_set_1) != len(score_set_2):
        print('Number of scores in each set must be equal. Returning (-1, -1).')
        return -1, -1
    else:
        score_diffs = [score_set_2[i] - score_set_1[i] for i in range(len(score_set_1))]

        if all([diff == 0 for diff in score_diffs]):
            print('No difference in score values passed in. Returning (0, 0).')
            return 0, 0

        stat, p = ttest_rel(score_set_1, score_set_2, 
                            alternative=alternative_hypothesis)

        return stat, p

In [10]:
def get_critical_value_for_improvement(sig_level, dof):
    return t.ppf(1 - sig_level, dof)

def get_t_value_paired_test(value_list_1, value_list_2, verbosity=0):
    if len(value_list_1) == len(value_list_2):
        diffs = [value_list_2[i] - value_list_1[i] for i in range(len(value_list_1))]
        diffs_mean = np.mean(diffs)
        diffs_sd = np.sqrt(sum([(diff - diffs_mean)**2 for diff in diffs]) / (len(diffs) + 1))
        t_val = diffs_mean / (diffs_sd / np.sqrt(len(diffs)))
        if verbosity > 0:
            print('mean of diffs = %.4f' % diffs_mean)
            print('sd of diffs = %.4f' % diffs_sd)
            print('t = %.4f' % t_val)
        return t_val
    else:
        print('Paired sample t-test requires input value lists to be the same size. Returning None.')
        return None
    
def check_if_new_scores_better(original_scores, new_scores, sig_level, verbosity=0):
    critical_val = get_critical_value_for_improvement(sig_level, len(original_scores) - 1)
    t_val = get_t_value_paired_test(original_scores, new_scores)
    new_scores_better = (t_val > critical_val)
    if verbosity > 0:
        print('critical value = %.4f' % critical_val)
    return new_scores_better

## Iterate over features, adding to candidate feature set if significantly beneficial to model performance

In [11]:
def logreg_feature_selection_v2(
    df,
    all_features_to_test,
    initial_features,
    selection_metric,
    random_state_list=[0, 42, 343],
    p_threshold=0.05,
    verbosity=0
):
    n_passes = len(random_state_list)
    
    initial_scores = train_logreg_cv_from_feature_set(df, initial_features, selection_metric)
    current_scores = initial_scores.copy()
    current_features = initial_features.copy()

    for n in range(n_passes):
        random_state = random_state_list[n]
        features_tested_count = 0

        # Initialize list of features to try in current iteration of loop
        features_to_test = all_features_to_test.copy()
        for feature in current_features:
            if feature in features_to_test:
                features_to_test.remove(feature)

        # Test if adding feature to feature set improves model performance
        for new_feature in features_to_test:
            temp_features = current_features + [new_feature]
            temp_scores = train_logreg_cv_from_feature_set(df, temp_features, scoring_metric=selection_metric)
            # temp_stat, temp_p = cv_auc_hypothesis_test(current_scores, temp_scores, 'less')
            
            new_scores_better = check_if_new_scores_better(current_scores, temp_scores, p_threshold, verbosity=verbosity)
    
            if new_scores_better:
                print('\n' + '~'*30)
                print('Old mean score: %3f' % np.mean(current_scores))
                print('New mean score: %3f' % np.mean(temp_scores))
                print('Adding ' + new_feature + ' to feature set.')
                print('~'*30 + '\n')
                print('Checked feature %d of %d total features (pass %d of %d).' % (features_tested_count,
                                                                                    len(features_to_test),
                                                                                    n,
                                                                                    n_passes))
                current_scores = temp_scores
                current_features = temp_features
    
            # if temp_p < p_threshold and np.mean(current_scores) < np.mean(temp_scores):
            #     print('\n' + '~'*30)
            #     print('Old mean score: %3f' % np.mean(current_scores))
            #     print('New mean score: %3f' % np.mean(temp_scores))
            #     print('Adding ' + new_feature + ' to feature set.')
            #     print('~'*30 + '\n')
            #     print('Checked feature %d of %d total features (pass %d of %d).' % (features_tested_count,
            #                                                                         len(features_to_test),
            #                                                                         pass_num,
            #                                                                         n_passes))
            #     current_scores = temp_scores
            #     current_features = temp_features

            features_tested_count += 1
            pass_num = n + 1

    return current_features

def svc_feature_selection_v2(
    df,
    all_features_to_test,
    initial_features,
    selection_metric,
    outfile_name='',
    max_iter=1000,
    samples_to_use=10000,
    sampling_seed=343,
    random_state_list=[0, 42, 343],
    p_threshold=0.05,
    svc_temp_results_dir='svc_temp_results/'
):
    n_passes = len(random_state_list)
    
    initial_scores = train_svc_cv_from_feature_set(
        df, 
        initial_features, 
        selection_metric, 
        max_iter=max_iter
    )
    current_scores = initial_scores.copy()
    current_features = initial_features.copy()
    
    features_tested = []
    
    start_dt_str = datetime.now().strftime('%Y%m%d-%H%M%S')
    
    # Get stratified sample
    total_dataset_size = len(df)
    positive_class_size = len(df[df['aki_72hrs_any']==1])
    positive_samples_to_use = int(np.round((positive_class_size / total_dataset_size) * samples_to_use))
    negative_samples_to_use = samples_to_use - positive_samples_to_use
    positive_samples = df[df['aki_72hrs_any']==1].sample(n = positive_samples_to_use)
    negative_samples = df[df['aki_72hrs_any']==0].sample(n = negative_samples_to_use)
    training_sample = pd.concat([positive_samples, negative_samples])

    for n in range(n_passes):
        random_state = random_state_list[n]
        features_tested_count = 0

        # Initialize list of features to try in current iteration of loop
        features_to_test = all_features_to_test.copy()
        for feature in current_features:
            if feature in features_to_test:
                features_to_test.remove(feature)

        # Test if adding feature to feature set improves model performance
        for new_feature in features_to_test:
            added_new_feature_to_candidates = False
            temp_features = current_features + [new_feature]
            temp_scores = train_svc_cv_from_feature_set(
                training_sample, 
                temp_features, 
                scoring_metric=selection_metric,
                max_iter=max_iter
            )
            # temp_stat, temp_p = cv_auc_hypothesis_test(current_scores, temp_scores, 'less')
            
            new_scores_better = check_if_new_scores_better(current_scores, temp_scores, p_threshold, verbosity=1)
    
            if new_scores_better:
                print('\n' + '~'*30)
                print('Old mean score: %3f' % np.mean(current_scores))
                print('New mean score: %3f' % np.mean(temp_scores))
                print('Adding ' + new_feature + ' to feature set.')
                print('~'*30 + '\n')
                print('Checked feature %d of %d total features (pass %d of %d).' % (features_tested_count,
                                                                                    len(features_to_test),
                                                                                    n,
                                                                                    n_passes))
                current_scores = temp_scores
                current_features = temp_features
                added_new_feature_to_candidates = True

            features_tested_count += 1
            pass_num = n + 1
            
            print('Checked feature %d of %d total features (pass %d of %d).' % (features_tested_count,
                                                                                len(features_to_test),
                                                                                n,
                                                                                n_passes))
            
            features_tested.append(new_feature)
            
            temp_dict = {
                'pass_num': pass_num,
                'features_tested': features_tested,
                'most_recent_feature_tested': new_feature,
                'most_recent_feature_added_to_candidates': added_new_feature_to_candidates,
                'current_candidate_feature_list': current_features,
                'current_scores': current_scores,
                'start_dt': start_dt_str,
                'all_features_to_test': all_features_to_test,
                'selection_metric': selection_metric,
                'random_states_remaining': random_state_list[n:]
            }
            
            datetime_str = datetime.now().strftime('%Y%m%d-%H%M%S')
            with open('pickle/candidate_features/' + svc_temp_results_dir + datetime_str + outfile_name + '.pickle', 'wb') as outfile:
                pickle.dump(temp_dict, outfile)
            

    return current_features

## Continue iterative feature set growth from temp file

In [12]:
def continue_candidate_feature_set_growth(
    temp_results_filename,
    selection_metric,
    temp_outfile_name,
    final_outfile_name,
    train_cohort,
    final_results_dir='pickle/candidate_features/',
    temp_results_dir='pickle/candidate_features/svc_temp_results_v3/',
    max_iter=2500,
    p_threshold=0.10
):
    with open(temp_results_dir + temp_results_filename, 'rb') as infile:
        temp_results_dict = pickle.load(infile)
        
    features_remaining_to_test = [
        feature for feature in temp_results_dict['all_features_to_test'] if (
            (feature not in temp_results_dict['features_tested']) & \
            (feature not in temp_results_dict['current_candidate_feature_list'])
        )
    ]
    features_to_start_with = temp_results_dict['current_candidate_feature_list']
    
    print('Remaining features to test count: %d' % len(features_remaining_to_test))
    print('Starting process...')
    
    candidate_feature_set = svc_feature_selection_v2(
        train_cohort,
        features_remaining_to_test,
        features_to_start_with,
        selection_metric,
        temp_outfile_name,
        max_iter=max_iter,
        random_state_list=temp_results_dict['random_states_remaining'],
        p_threshold=p_threshold
    )
    
    with open(final_results_dir + final_outfile_name, 'wb') as outfile:
        pickle.dump(candidate_feature_set, outfile)

# Perform feature selection

## Create list of features to test

In [13]:
features_to_test = [
    feature for feature in prepare_data.get_feature_cols(train_cohort_ll) if \
            (feature not in removed_features) & ('imputation_median' not in feature)
]

In [14]:
len(features_to_test)

157

## Create list of initial candidate features

Use top 2 features for each metric.

In [15]:
combined_single_feature_metrics.head()

,input_feature,auroc_svc,auprc_svc,auroc_logreg,auprc_logreg
0,ageatadmission,0.383238,0.023236,0.600715,0.109145
1,ALBUMIN_max,0.499085,0.032375,0.499244,0.032555
2,ALBUMIN_mean,0.501799,0.028491,0.498402,0.033431
3,ALBUMIN_median,0.498230,0.033083,0.498411,0.033281
4,ALBUMIN_min,0.501599,0.028378,0.498568,0.034995


In [16]:
initial_feature_set = []

for metric in ['auroc_logreg', 'auprc_logreg', 'auroc_svc', 'auprc_svc']:
    for feature in combined_single_feature_metrics.sort_values(metric, ascending=False).head(2)['input_feature']:
        if feature not in initial_feature_set:
            initial_feature_set.append(feature)
            
print('\n'.join(initial_feature_set))

CREATININE_median
CREATININE_mean
sbp_median
sbp_mean
map_mean
mbp_min
mbp_max


## Logistic regression

### Latest lab imputation

#### AUROC

In [17]:
auroc_logreg_features = logreg_feature_selection_v2(
    train_cohort_ll,
    features_to_test,
    initial_feature_set,
    'roc_auc',
    random_state_list=[0, 42, 343],
    p_threshold=0.05
)


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.852421
New mean score: 0.853769
Adding fio2_mean to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 1 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.853769
New mean score: 0.854711
Adding fio2_max to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 2 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.854711
New mean score: 0.856130
Adding mbp_median to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 6 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.856130
New mean score: 0.858233
Adding pox_mean to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 7 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.858233
New mean score: 0.858842
Adding pulse_median to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 13 of 151 total features (p

In [18]:
print('\n'.join(initial_feature_set))

CREATININE_median
CREATININE_mean
sbp_median
sbp_mean
map_mean
mbp_min
mbp_max


In [19]:
with open('pickle/candidate_features/logreg_latest_lab_auroc_candidate_feature_set.pickle', 'wb') as outfile:
    pickle.dump(auroc_logreg_features, outfile)

#### AUPRC

In [20]:
auprc_logreg_features = logreg_feature_selection_v2(
    train_cohort_ll,
    features_to_test,
    initial_feature_set,
    'average_precision',
    random_state_list=[0, 42, 343],
    p_threshold=0.05
)


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.280512
New mean score: 0.280953
Adding map_min to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 3 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.280953
New mean score: 0.283973
Adding pox_min to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 8 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.283973
New mean score: 0.285156
Adding pulse_mean to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 10 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.285156
New mean score: 0.285376
Adding pulse_median to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 13 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.285376
New mean score: 0.289388
Adding resp_rate_mean to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 14 of 151 total featur

In [21]:
with open('pickle/candidate_features/logreg_latest_lab_auprc_candidate_feature_set.pickle', 'wb') as outfile:
    pickle.dump(auprc_logreg_features, outfile)

### Median imputation

#### AUROC

In [22]:
auroc_logreg_features_med_imp = logreg_feature_selection_v2(
    train_cohort_med_imp,
    features_to_test,
    initial_feature_set,
    'roc_auc',
    random_state_list=[0, 42, 343],
    p_threshold=0.05
)


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.752339
New mean score: 0.757063
Adding fio2_mean to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 1 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.757063
New mean score: 0.761608
Adding mbp_mean to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 5 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.761608
New mean score: 0.764035
Adding pox_mean to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 7 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.764035
New mean score: 0.766006
Adding pulse_max to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 12 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.766006
New mean score: 0.769048
Adding pulse_median to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 13 of 151 total features (p

In [23]:
with open('pickle/candidate_features/logreg_median_auroc_candidate_feature_set.pickle', 'wb') as outfile:
    pickle.dump(auroc_logreg_features_med_imp, outfile)

#### AUPRC

In [24]:
auprc_logreg_features_med_imp = logreg_feature_selection_v2(
    train_cohort_med_imp,
    features_to_test,
    initial_feature_set,
    'average_precision',
    random_state_list=[0, 42, 343],
    p_threshold=0.05
)


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.144673
New mean score: 0.147025
Adding mbp_mean to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 5 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.147025
New mean score: 0.154571
Adding MAGNESIUM_mean to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 40 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.154571
New mean score: 0.182954
Adding MCV_mean to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 41 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.182954
New mean score: 0.191035
Adding NEUTRO_PCT_mean to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 42 of 151 total features (pass 0 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.191035
New mean score: 0.194867
Adding PH_mean to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 44 of 151 total fea

In [25]:
with open('pickle/candidate_features/logreg_median_auprc_candidate_feature_set_v2.pickle', 'wb') as outfile:
    pickle.dump(auprc_logreg_features_med_imp, outfile)

## SVC

### Latest lab imputation

#### Load candidate features from logistic regression testing

In [26]:
with open('pickle/candidate_features/logreg_latest_lab_candidate_feature_dict_v2.pickle', 'rb') as infile:
    logreg_candidate_feature_set_dict = pickle.load(infile)

#### AUROC

In [27]:
with open('pickle/candidate_features/logreg_latest_lab_auroc_candidate_feature_set.pickle', 'rb') as infile:
    ll_roc_candidate_features = pickle.load(infile)

In [28]:
warnings.filterwarnings("ignore")

In [3]:
auroc_svc_features = svc_feature_selection_v2(
    train_cohort_ll,
    features_to_test,
    ll_roc_candidate_features,
    'roc_auc',
    outfile_name='auroc_latest_lab',
    max_iter=2500,
    samples_to_use=10000,
    sampling_seed=343,
    random_state_list=[343],
    p_threshold=0.10
)

In [35]:
with open('pickle/candidate_features/svc_latest_lab_auroc_candidate_features.pickle', 'wb') as outfile:
    pickle.dump(auroc_svc_features, outfile)

#### AUPRC

In [36]:
with open('pickle/candidate_features/logreg_latest_lab_auprc_candidate_feature_set.pickle', 'rb') as infile:
    ll_prc_candidate_features = pickle.load(infile)

In [4]:
auprc_ll_v2_svc_features = svc_feature_selection_v2(
    train_cohort_ll,
    features_to_test,
    ll_prc_candidate_features,
    'average_precision',
    outfile_name='auprc_latest_lab',
    max_iter=2500,
    samples_to_use=10000,
    sampling_seed=343,
    random_state_list=[343],
    p_threshold=0.10
)

In [38]:
with open('pickle/candidate_features/svc_latest_lab_auprc_candidate_features.pickle', 'wb') as outfile:
    pickle.dump(auprc_ll_v2_svc_features, outfile)

### Median imputation

#### AUROC

Load logistic regression selected features

In [39]:
with open('pickle/candidate_features/logreg_median_auroc_candidate_feature_set.pickle', 'rb') as infile:
    auroc_logreg_features_med_imp = pickle.load(infile)

In [5]:
auroc_svc_features_med_imp = svc_feature_selection_v2(
    train_cohort_med_imp,
    features_to_test,
    auroc_logreg_features_med_imp,
    'roc_auc',
    outfile_name='auroc_med_imp',
    max_iter=2500,
    samples_to_use=10000,
    sampling_seed=343,
    random_state_list=[343],
    p_threshold=0.10
)

In [41]:
with open('pickle/candidate_features/svc_median_imp_auproc_candidate_features.pickle', 'wb') as outfile:
    pickle.dump(auroc_svc_features_med_imp, outfile)

#### AUPRC

Load logistic regression selected features

In [42]:
with open('pickle/candidate_features/logreg_median_auprc_candidate_feature_set.pickle', 'rb') as infile:
    auprc_logreg_features_med_imp = pickle.load(infile)

In [6]:
auprc_svc_features_med_imp = svc_feature_selection_v2(
    train_cohort_med_imp,
    features_to_test,
    auprc_logreg_features_med_imp,
    'average_precision',
    outfile_name='auprc_med_imp',
    max_iter=2500,
    samples_to_use=10000,
    sampling_seed=343,
    random_state_list=[343],
    p_threshold=0.10
)

In [44]:
with open('pickle/candidate_features/svc_median_auprc_candidate_feature_set.pickle', 'wb') as outfile:
    pickle.dump(auprc_svc_features_med_imp, outfile)